In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, Lambda
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# 1. Data Preparation
corpus = [
    "Deep learning is a subset of machine learning.",
    "Neural networks are inspired by the human brain.",
    "Convolutional neural networks excel at image classification.",
    "Word embeddings capture semantic meaning in natural language processing."
]

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
word2id = tokenizer.word_index
id2word = {v: k for k, v in word2id.items()}
vocab_size = len(word2id) + 1
embed_size = 10
window_size = 2

# 2. Generate Context-Target Pairs
sequences = tokenizer.texts_to_sequences(corpus)
X, Y = [], []

for seq in sequences:
    for i, target_word in enumerate(seq):
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(seq):
                context.append(seq[j])
        if len(context) == window_size * 2:
            X.append(context)
            Y.append(target_word)

X = np.array(X)
Y = to_categorical(Y, num_classes=vocab_size)

# 3. Build CBOW Model
cbow = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embed_size, input_length=window_size*2),
    Lambda(lambda x: tf.reduce_mean(x, axis=1)),
    Dense(vocab_size, activation='softmax')
])

cbow.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 4. Train Model
cbow.fit(X, Y, epochs=200, verbose=0)
print("CBOW Training Complete!")

# 5. Test Prediction on a Sample Context
sample_context = X[0].reshape(1, -1)
pred_id = np.argmax(cbow.predict(sample_context), axis=1)[0]
context_words = [id2word[idx] for idx in X[0]]
print(f"Context Words: {context_words} --> Predicted Target Word: '{id2word[pred_id]}'")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


CBOW Training Complete!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Context Words: ['deep', 'learning', 'a', 'subset'] --> Predicted Target Word: 'is'
